═══════════════════════════════════════════════════════════════════
Paper 2 — AI-Driven Intervention Portfolio Optimisation for
          Industrial Emissions Reduction

Authors: Idris Alugo
Journal: (target)
Zenodo:  (DOI to be assigned)

Entry point: run this notebook top-to-bottom to reproduce
             all figures and tables in the paper.
═══════════════════════════════════════════════════════════════════

In [ ]:
# ── Cell 0: Environment check ─────────────────────────────────────
import sys, importlib
print(f"Python {sys.version}")
required = ["numpy","pandas","sklearn","scipy","matplotlib"]
for pkg in required:
    try:    importlib.import_module(pkg); print(f"  ✓ {pkg}")
    except ImportError: print(f"  ✗ {pkg}  MISSING")

In [ ]:
# ── Cell 1: Imports & logging ─────────────────────────────────────
import logging, warnings, os
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))        # adjust if needed

In [ ]:
from config import (
    DATA_PATH, ESG_PATH, INTERVENTION_PATH, FIGURES_DIR, OUTPUTS_DIR,
    K_CLUSTERS, CLUSTER_SEED, N_INIT,
    BUDGET_TOTAL, BUDGET_PER_FAC, RANDOM_SEED,
)
from src.clustering      import (build_facility_profile, run_kmeans,
                                  assign_tiers, elbow_silhouette_scan)
from src.interventions   import (load_interventions, filter_by_tier,
                                  compute_cost_effectiveness)
from src.optimisation    import (greedy_portfolio, lp_portfolio,
                                  portfolio_summary, lp_min_budget)
from src.visualization_p2 import (plot_figure6, plot_elbow_silhouette,
                                   plot_cost_effectiveness,
                                   plot_abatement_frontier)

In [ ]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)-8s %(name)s | %(message)s",
    datefmt="%H:%M:%S",
)
logger = logging.getLogger("paper2_pipeline")
np.random.seed(RANDOM_SEED)
logger.info("Imports complete.")

In [ ]:
# ── Cell 2: Load Paper 1 outputs ──────────────────────────────────
# risk_df.csv is the output of Paper 1 (01_run_pipeline.ipynb cell 6)
# Columns: Facility, Prediction, Target_2030, Uncertainty_sigma,
#          P_i_percent, RiskLevel
risk_df = pd.read_csv(DATA_PATH)
logger.info("Paper 1 risk data loaded: %d facilities", len(risk_df))

In [ ]:
# Normalise column names from Paper 1 output to Paper 2 expected schema
risk_df = risk_df.rename(columns={
    "PredictionEnsemble":      "Prediction",
    "ProbMeetTargetEnsemble":  "P_i_percent",
    "RiskLevelEnsemble":       "RiskLevel",
    "Target2030":              "Target_2030",
    "UncertaintyEnsemble":     "Uncertainty_sigma",
})
logger.info("Paper 1 risk data loaded: %d facilities | columns: %s",
            len(risk_df), risk_df.columns.tolist())

In [ ]:
# Also load the raw emissions series for profile building
# (same esgdata.csv used in Paper 1)
raw_df = pd.read_csv(ESG_PATH, parse_dates=["Date"])
logger.info("Raw emissions data: %s", raw_df.shape)

In [ ]:
print(raw_df.columns.tolist())
print(raw_df.dtypes)

In [ ]:
# ── Filter to study facilities ────────────────────────────────────
# Derive from risk_df so this cell works regardless of how many
# facilities Paper 1 produces (currently 30).
STUDY_FACILITIES = risk_df["Facility"].tolist()

raw_df = raw_df[raw_df["Facility"].isin(STUDY_FACILITIES)].reset_index(drop=True)

logger.info("Filtered to %d study facilities: %d risk rows, %d raw rows",
            len(STUDY_FACILITIES), len(risk_df), len(raw_df))

In [ ]:
# ── Cell 3: Facility Profiling ────────────────────────────────────
profile = build_facility_profile(raw_df, target_col="Emissions_tCO2")
profile.to_csv(os.path.join(OUTPUTS_DIR, "facility_profiles.csv"))
logger.info("Facility profiles:\n%s", profile.describe().round(2).to_string())

In [ ]:
import importlib, src.clustering, src.visualization_p2, src.optimisation
importlib.reload(src.clustering)
importlib.reload(src.visualization_p2)
importlib.reload(src.optimisation)
from src.clustering      import (build_facility_profile, run_kmeans,
                                  assign_tiers, elbow_silhouette_scan)
from src.visualization_p2 import (plot_figure6, plot_elbow_silhouette,
                                   plot_cost_effectiveness,
                                   plot_abatement_frontier)
from src.optimisation    import (greedy_portfolio, lp_portfolio,
                                  portfolio_summary, lp_min_budget)

In [ ]:
# ── Cell 4: Elbow / Silhouette Scan (Appendix Fig S1) ────────────
scan_df = elbow_silhouette_scan(profile, k_range=range(2, 8),
                                 seed=CLUSTER_SEED, n_init=N_INIT)
print(scan_df.to_string(index=False))
plot_elbow_silhouette(scan_df, FIGURES_DIR)
scan_df.to_csv(os.path.join(OUTPUTS_DIR, "kmeans_scan.csv"), index=False)

In [ ]:
# ── Cell 5: K-means Clustering (K=3) ─────────────────────────────
cluster_result = run_kmeans(profile, k=K_CLUSTERS,
                             seed=CLUSTER_SEED, n_init=N_INIT)
tier_df = assign_tiers(cluster_result)
tier_df.to_csv(os.path.join(OUTPUTS_DIR, "facility_tiers.csv"))

# Drop TierLabel if already present (e.g. from a previous run saved in CSV)
if "TierLabel" in risk_df.columns:
    risk_df = risk_df.drop(columns=["TierLabel"])

# Join TierLabel into risk_df so all downstream outputs carry it
risk_df = risk_df.merge(
    tier_df[["TierLabel"]].reset_index().rename(columns={"index": "Facility"}),
    on="Facility", how="left"
)

# Persist updated risk_df (with TierLabel) back to disk
risk_df.to_csv(DATA_PATH, index=False)
logger.info("risk_df saved with TierLabel: %s", DATA_PATH)

In [ ]:
print("\nTier distribution:")
print(tier_df["TierLabel"].value_counts().to_string())
print(f"\nGlobal Silhouette = {cluster_result['silhouette_global']:.4f}")

In [ ]:
# ── Cell 6: Figure 6 ─────────────────────────────────────────────
fig6_path = plot_figure6(cluster_result, tier_df, FIGURES_DIR, scan_df=scan_df)
print(f"Figure 6 saved → {fig6_path}")

In [ ]:
# ── Cell 7: Load Intervention Catalogue ──────────────────────────
interventions = load_interventions(INTERVENTION_PATH)
print(interventions[["InterventionID","Name","Category",
                      "Cost_EUR","AbatementRate"]].to_string(index=False))

In [ ]:
# ── Cell 8: Build Per-Facility Candidate List ─────────────────────
# Use PredictionEnsemble (forward-looking) as facility baseline for
# abatement calculations, NOT historical mean_emissions.
# This ensures cost-per-tonne reflects the actual current trajectory,
# so large facilities with high predictions correctly dominate the LP.
risk_lookup = risk_df.set_index("Facility")["Prediction"]

candidates_list = []
for fac in tier_df.index:
    tier      = int(tier_df.loc[fac, "Tier"])
    baseline  = float(risk_lookup.loc[fac]) if fac in risk_lookup.index \
                else tier_df.loc[fac, "mean_emissions"]
    avail     = filter_by_tier(interventions, tier)
    costed    = compute_cost_effectiveness(avail, facility_baseline=baseline)
    costed["Facility"]  = fac
    costed["Tier"]      = tier
    costed["TierLabel"] = tier_df.loc[fac, "TierLabel"]
    candidates_list.append(costed)

In [ ]:
all_candidates = pd.concat(candidates_list, ignore_index=True)
logger.info("Total candidate interventions: %d", len(all_candidates))
all_candidates.to_csv(os.path.join(OUTPUTS_DIR, "all_candidates.csv"), index=False)

In [ ]:
# ── Cell 9: Greedy Portfolio ──────────────────────────────────────
greedy_df = greedy_portfolio(all_candidates.copy(),
                              budget=BUDGET_TOTAL,
                              per_fac_cap=BUDGET_PER_FAC)
greedy_summary = portfolio_summary(greedy_df)
print("\nGreedy Portfolio Summary:")
print(greedy_summary.to_string(index=False))
greedy_summary.to_csv(os.path.join(OUTPUTS_DIR, "greedy_portfolio.csv"), index=False)

In [ ]:
# ── Cell 10: LP Portfolio ─────────────────────────────────────────
lp_df = lp_portfolio(all_candidates.copy(),
                      budget=BUDGET_TOTAL,
                      per_fac_cap=BUDGET_PER_FAC,
                      risk_df=risk_df)
lp_summary = portfolio_summary(lp_df)
print("\nLP Portfolio Summary:")
print(lp_summary.to_string(index=False))
lp_summary.to_csv(os.path.join(OUTPUTS_DIR, "lp_portfolio.csv"), index=False)

# Portfolio P across ALL facilities (on-track kept at P_pre)
p_lp  = lp_df.groupby("Facility")["P_post"].first()
p_all = risk_df.set_index("Facility")["P_i_percent"].copy()
p_all.update(p_lp)
print(f"\nBudget used:          €{(lp_df['x_lp']*lp_df['Cost_EUR']).sum():>12,.0f}")
print(f"Portfolio P BEFORE:    {risk_df['P_i_percent'].mean():.4f}")
print(f"Portfolio P AFTER LP:  {p_all.mean():.4f}")
print(f"LP improvement:       {p_all.mean()-risk_df['P_i_percent'].mean():>+.4f}")


In [ ]:
# ── Cell 11: Merge with Risk Data for Table 3 ─────────────────────
table3 = tier_df.merge(
    lp_summary.rename(columns={"Facility":"Facility"}),
    left_index=True, right_on="Facility", how="left"
).merge(
    risk_df[["Facility","Prediction","P_i_percent","RiskLevel"]],
    on="Facility", how="left"
).sort_values(["Tier","Prediction"])
print("\nTable 3 (LP + Risk):")
display_cols = ["Facility","TierLabel","Total_Cost_EUR",
                "Total_Abatement","RiskLevel","P_i_percent"]
if "P_post" in table3.columns:
    display_cols.append("P_post")
print(table3[display_cols].to_string(index=False))
table3.to_csv(os.path.join(OUTPUTS_DIR, "table3_portfolio_risk.csv"), index=False)

In [ ]:
# ── Cell 12: Figure 8 – Cost-Effectiveness by Tier ───────────────
# Attach TierLabel to per-facility LP summary for the bar chart
ce_df = lp_summary.merge(
    tier_df[["TierLabel"]].reset_index(), on="Facility", how="left"
)
plot_cost_effectiveness(ce_df, FIGURES_DIR)

In [ ]:
# ── Cell 13: Figure 9 – Abatement Frontier ───────────────────────
budgets    = np.linspace(100_000, 2_000_000, 20)
frontier   = []
for b in budgets:
    tmp = lp_portfolio(all_candidates.copy(), budget=b,
                       per_fac_cap=BUDGET_PER_FAC, risk_df=risk_df)
    abatement = (tmp["x_lp"] * tmp["AbatementVolume_tCO2"]).sum()
    frontier.append({"Budget_EUR": b, "Abatement_tCO2": abatement})
frontier_df = pd.DataFrame(frontier)
plot_abatement_frontier(frontier_df, FIGURES_DIR)
frontier_df.to_csv(os.path.join(OUTPUTS_DIR, "abatement_frontier.csv"), index=False)

In [ ]:
# ── Cell 14: Minimum Budget to Achieve Portfolio P ≥ 0.80 ─────
P_TARGET = 0.80

opt_budget, detail_df, portfolio_p = lp_min_budget(
    all_candidates.copy(), risk_df, p_target=P_TARGET
)

print("=" * 62)
print("MINIMUM BUDGET LP SOLUTION")
print("=" * 62)
print(f"OPTIMAL_BUDGET_EUR      = {opt_budget:>12,.0f}")
print(f"Portfolio P achieved    = {portfolio_p:.4f}  (target ≥ {P_TARGET})")
n_int = int((detail_df.groupby("Facility")["Alloc_EUR"].sum() > 1e-3).sum())
print(f"Facilities intervened   = {n_int} / {len(risk_df)}")
print()

alloc_df = detail_df[detail_df["Alloc_EUR"] > 1e-3].copy()
alloc_df["Alloc_EUR"] = alloc_df["Alloc_EUR"].round(0).astype(int)
print("Allocation detail (active interventions only):")
print(alloc_df[["Facility", "InterventionID", "InterventionName",
                 "Alloc_EUR", "Alloc_Abatement_tCO2",
                 "pred_pre", "pred_post", "p_pre", "p_post"]].to_string(index=False))

detail_df.to_csv(os.path.join(OUTPUTS_DIR, "lp_optimal_budget.csv"), index=False)
logger.info("Saved lp_optimal_budget.csv")

print()
print("=" * 62)
print("BUDGET SCENARIOS FOR RL EXPERIMENTS")
print(f"  (fractions of BUDGET_TOTAL = €{BUDGET_TOTAL:,.0f})")
print("=" * 62)
print(f"  {'Scenario':<36} {'Budget (EUR)':>14}   {'Portfolio P':>11}")
print(f"  {'-'*36} {'-'*14}   {'-'*11}")

# Scenarios as fractions of BUDGET_TOTAL so RL experiments use the
# same budget range.  Portfolio P merges LP results onto all risk_df
# facilities so on-track facilities are counted at their P_pre.
for label, frac in [
    ("100% — €1.5M full budget",    1.0),
    ("80%  — moderate scarcity",     0.8),
    ("70%  — tight",                 0.7),
    ("60%  — severe scarcity",       0.6),
]:
    b = BUDGET_TOTAL * frac
    tmp = lp_portfolio(
        all_candidates.copy(),
        budget=b,
        per_fac_cap=BUDGET_PER_FAC,
        risk_df=risk_df,
    )
    p_lp  = tmp.groupby("Facility")["P_post"].first()
    p_all = risk_df.set_index("Facility")["P_i_percent"].copy()
    p_all.update(p_lp)
    p_scen = p_all.mean()
    print(f"  {label:<36} {b:>14,.0f}   {p_scen:>11.4f}")


In [ ]:
logger.info("Pipeline complete. All figures in %s", FIGURES_DIR)
logger.info("All tables  in %s", OUTPUTS_DIR)

In [ ]:
# ── Clean summary of the min-budget LP solution ───────────────────
# detail_df is already in memory from Cell 14; no need to re-read disk.

# Facility-level summary (deduplicate – p_post is the same for all
# interventions belonging to the same facility)
fac_sum = detail_df.groupby("Facility").agg(
    total_cost=("Alloc_EUR",            "sum"),
    pred_pre  =("pred_pre",             "first"),
    pred_post =("pred_post",            "first"),
    p_pre     =("p_pre",                "first"),
    p_post    =("p_post",               "first"),
).reset_index()

print(f"{'Facility':<8} {'Cost':>10} {'pred_pre':>10} {'pred_post':>10} "
      f"{'P_pre':>7} {'P_post':>7} {'ΔP':>7}")
print("-" * 65)
for _, r in fac_sum.sort_values("total_cost", ascending=False).iterrows():
    print(f"{r['Facility']:<8} {r['total_cost']:>10,.0f} {r['pred_pre']:>10.1f} "
          f"{r['pred_post']:>10.1f} {r['p_pre']:>7.3f} {r['p_post']:>7.3f} "
          f"{r['p_post'] - r['p_pre']:>+7.3f}")

print(f"\nTotal budget used:     {fac_sum['total_cost'].sum():>10,.0f} EUR")
print(f"Portfolio P BEFORE:    {fac_sum['p_pre'].mean():>10.4f}")
print(f"Portfolio P AFTER LP:  {fac_sum['p_post'].mean():>10.4f}")
print(f"LP improvement:        {fac_sum['p_post'].mean() - fac_sum['p_pre'].mean():>+10.4f}")
print(f"\nFacilities with gap>0 that got budget:")
gap_facs = fac_sum[fac_sum["pred_post"] < fac_sum["pred_pre"]]
print(gap_facs[["Facility", "total_cost", "p_pre", "p_post"]].to_string(index=False))